# Tale Role — QLoRA mechanics (7B)

Train **our** mechanics adapter on the synthetic JSONL in this repo. No paid OpenAI/Anthropic/HF Inference API.

- Base: `Qwen/Qwen2.5-7B-Instruct` (matches `llm/mechanics/card.json`)
- Export only the LoRA adapter. Download it to `$TALEROLE_ADAPTER_DIR/mechanics` on your machine.
- Never paste player tables, emails, or OTP codes into this notebook.
- Mechanics JSON must **not** emit dice, HP, or turn order. The Go engine writes those.

Storyteller can reuse this 7B recipe as a stand-in. The 32B card is for a later GPU host — not this Colab.

In [ ]:
!pip install -q "transformers>=4.44" "datasets>=2.20" "peft>=0.12" "bitsandbytes>=0.43" "accelerate>=0.33" trl

Upload `llm/datasets/synthetic/mechanics.jsonl` (or clone the public repo). Do not upload production exports.

In [ ]:
from pathlib import Path
import json
from datasets import Dataset

path = Path("mechanics.jsonl")
assert path.exists(), "Upload llm/datasets/synthetic/mechanics.jsonl first"
rows = [json.loads(line) for line in path.read_text(encoding="utf-8").splitlines() if line.strip()]

def to_text(row):
    user = json.dumps(row["input"], ensure_ascii=False)
    assistant = json.dumps(row["output"], ensure_ascii=False)
    return {
        "text": (
            "<|im_start|>system\nReturn only mechanic JSON. Never dice, HP, or turn order.<|im_end|>\n"
            f"<|im_start|>user\n{user}<|im_end|>\n"
            f"<|im_start|>assistant\n{assistant}<|im_end|>\n"
        )
    }

ds = Dataset.from_list([to_text(r) for r in rows])
print(len(ds), "rows")

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TrainingArguments
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer

base = "Qwen/Qwen2.5-7B-Instruct"
bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)
tok = AutoTokenizer.from_pretrained(base, trust_remote_code=True)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token
model = AutoModelForCausalLM.from_pretrained(
    base, quantization_config=bnb, device_map="auto", trust_remote_code=True
)
model = prepare_model_for_kbit_training(model)
model = get_peft_model(
    model,
    LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, task_type="CAUSAL_LM", target_modules="all-linear"),
)
args = TrainingArguments(
    output_dir="mechanics-qlora",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    num_train_epochs=3,
    learning_rate=2e-4,
    logging_steps=1,
    fp16=True,
    report_to=[],
)
trainer = SFTTrainer(model=model, tokenizer=tok, train_dataset=ds, args=args, dataset_text_field="text", max_seq_length=512)
trainer.train()
trainer.save_model("mechanics-adapter")
tok.save_pretrained("mechanics-adapter")
print("Download the mechanics-adapter folder onto TALEROLE_ADAPTER_DIR/mechanics (private disk, not git).")